# 🔬 Notebook 3: Typeahead / Autocomplete — Deep Dive: Trie implementation, decay & updates

## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Build and query a trie — end to end

In [ ]:
from collections import defaultdict

class TrieNode:
    __slots__ = ("children","is_end","score","top_k")
    def __init__(self):
        self.children: dict[str, "TrieNode"] = {}
        self.is_end = False
        self.score = 0           # for this exact term
        self.top_k: list[tuple[str,int]] = []

class Trie:
    def __init__(self, k=5):
        self.root = TrieNode()
        self.k = k

    def add(self, term: str, freq: int = 1):
        node = self.root
        for ch in term:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True
        node.score += freq

    def _all_terms(self, node: TrieNode, prefix: str, out: list):
        if node.is_end: out.append((prefix, node.score))
        for ch, child in node.children.items():
            self._all_terms(child, prefix + ch, out)

    def precompute_top_k(self):
        # For each node, compute its top-K descendants.
        def dfs(node: TrieNode, prefix: str):
            terms: list[tuple[str,int]] = []
            if node.is_end: terms.append((prefix, node.score))
            for ch, child in node.children.items():
                terms += dfs(child, prefix + ch)
            node.top_k = sorted(terms, key=lambda x: -x[1])[: self.k]
            return terms
        dfs(self.root, "")

    def suggest(self, prefix: str):
        node = self.root
        for ch in prefix:
            if ch not in node.children: return []
            node = node.children[ch]
        return node.top_k

t = Trie(k=3)
for term, freq in [
    ("python", 900), ("python tutorial", 500), ("pyramid", 100),
    ("go", 300), ("golang", 200), ("google", 800), ("google docs", 400),
]:
    t.add(term, freq)
t.precompute_top_k()

print("suggest('py') →", t.suggest("py"))
print("suggest('goog') →", t.suggest("goog"))


## Deep dive 1 — memory & sharding

- **One trie for English** with 100M distinct terms, avg 12 chars: ~3 GB nodes.
- Fits on **one big machine**. But we need many replicas for QPS.
- For multi-language: one trie per language.
- For personalization: a *small* per-user trie merged with the global one at query time.

### Updating
Mutating a live trie concurrently is complex. Instead:
1. Build a new trie from query logs in a batch job.
2. Atomically swap the pointer (nodes read immutably after construction).
3. Old trie freed after outstanding requests drain.


## Deep dive 2 — ranking beyond frequency

Popularity alone causes staleness. Real systems blend:
- **Raw frequency** (last N days)
- **Trending score**: d(frequency)/dt
- **Personalization**: this user's previous queries, region
- **Spell correction**: if no prefix match, try edit-distance-1 variants

Freshness can be handled with a **decaying counter**: every hour, multiply counters by 0.99.
Trending terms rise fast; yesterday's news fades.

In [ ]:
# Tiny exponential decay example
import time
class DecayingCounter:
    def __init__(self, half_life_s=3600):
        self.half_life = half_life_s
        self.count = 0.0
        self.ts = time.time()
    def add(self, n=1):
        self._decay(); self.count += n
    def value(self):
        self._decay(); return self.count
    def _decay(self):
        now = time.time(); dt = now - self.ts
        if dt > 0:
            self.count *= 0.5 ** (dt / self.half_life); self.ts = now

c = DecayingCounter(half_life_s=0.5)
c.add(100); time.sleep(0.25); print("after 0.25s:", round(c.value(), 2))
time.sleep(0.5); print("after 0.75s:", round(c.value(), 2))
